# Homework 2 — Prompt Engineering and RAG for Machine Translation

**WMT16 EN↔TR | Qwen 2.5 7B Instruct | MAPS + RAG**

He et al., TACL 2024 — *Exploring Human-Like Translation Strategy with LLMs*

---

## Hücre Çalıştırma Sırası

1. **Setup** (A, B, C) — bir kez çalıştır
2. **Part 1** — veri yükle ve kaydet
3. **Part 2** — modeli yükle + benchmark
4. **Part 3** — zero-shot ve MAPS deneyleri
5. **Part 4** — RAG index + deney
6. **Part 5** — karşılaştırma ve COMET skorları

## A — Bağımlılıkları Kur

In [ ]:
# Tüm bağımlılıkları kur (ilk çalıştırmada ~5 dakika)
!pip install -q \
    transformers==4.45.0 \
    accelerate==0.27.0 \
    bitsandbytes==0.43.0 \
    datasets==2.20.0 \
    sentence-transformers==3.0.0 \
    faiss-cpu==1.7.4 \
    unbabel-comet==2.2.0 \
    pyyaml numpy pandas tqdm scipy

## B — Drive Mount ve Repo Clone

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_DIR = '/content/rag-mt-homework'
DRIVE_DIR = '/content/drive/MyDrive/RAG_Project'

# İlk kez clone, sonraki oturumlarda pull
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/mustafagalata/rag-mt-homework {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

# Drive'da proje dizinlerini oluştur
for d in ['data', 'indices', 'results', 'cache']:
    os.makedirs(f'{DRIVE_DIR}/{d}', exist_ok=True)

print('Dizinler hazır.')

## C — Import ve Config

In [ ]:
import sys, json, time, logging
from pathlib import Path
from tqdm.notebook import tqdm

sys.path.insert(0, REPO_DIR)

import yaml
with open(f'{REPO_DIR}/config.yaml') as f:
    cfg = yaml.safe_load(f)

# Path'leri Drive'a yönlendir
cfg['paths'] = {
    'data_dir':   f'{DRIVE_DIR}/data',
    'index_dir':  f'{DRIVE_DIR}/indices',
    'results_dir':f'{DRIVE_DIR}/results',
    'cache_dir':  f'{DRIVE_DIR}/cache',
}

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
print('Config yüklendi.')

---

## Part 1 — Dataset Selection and Preparation

In [ ]:
from src.data_loader import load_wmt16_splits, save_splits, load_splits_from_disk, print_sample_pairs
from pathlib import Path

data_dir = cfg['paths']['data_dir']

# Daha önce kaydedildiyse diskten yükle
if Path(f'{data_dir}/train.jsonl').exists():
    print('Diskten yükleniyor...')
    splits = load_splits_from_disk(data_dir)
else:
    print('WMT16 indiriliyor ve işleniyor...')
    splits = load_wmt16_splits(cfg, cache_dir=cfg['paths']['cache_dir'])
    save_splits(splits, data_dir)

print(f"Train: {len(splits['train']):,} | EN→TR test: {len(splits['test_en2tr']):,} | TR→EN test: {len(splits['test_tr2en']):,}")

In [ ]:
# Part 1.d: Örnek input-output çiftleri
print_sample_pairs(splits, n=5)

---

## Part 2 — Model Selection

In [ ]:
# GPU bilgisi
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

In [ ]:
from src.model import load_model_and_tokenizer, TranslationModel

model_raw, tokenizer = load_model_and_tokenizer(
    cfg['model']['name'],
    load_in_4bit=cfg['model']['load_in_4bit'],
    device_map=cfg['model']['device_map'],
    cache_dir=cfg['paths']['cache_dir'],
)
model = TranslationModel(model_raw, tokenizer, cfg)

In [ ]:
# ── Bölüm 7 (YOL_HARITASI): Ön Benchmark ──
# 50 cümle üzerinde 3 yaklaşım için süre ölç

from src.prompts.zero_shot import build_zero_shot_prompt, detect_language_from_pair
from src.prompts.maps_strategy import MAPSPipeline

benchmark_pairs = splits['test_en2tr'][:cfg['benchmark']['n_sentences']]
maps_pipeline = MAPSPipeline(model, cfg)

# Zero-shot benchmark
t0 = time.time()
_ = [model.generate(*build_zero_shot_prompt(p['en'], 'English', 'Turkish')) for p in benchmark_pairs[:10]]
zs_time = (time.time() - t0) / 10
print(f'Zero-shot: {zs_time:.1f} s/cümle (tahmini)')

# MAPS benchmark (1 cümle)
t0 = time.time()
_ = maps_pipeline.run(benchmark_pairs[0]['en'], 'English', 'Turkish')
maps_time = time.time() - t0
print(f'MAPS:      {maps_time:.1f} s/cümle')

# Tahminler
n_test = cfg['data']['test_subset_size']
print(f'\nToplam tahmin ({n_test} cümle/yön, 2 yön):')
print(f'  Zero-shot: {zs_time * n_test * 2 / 3600:.1f} saat')
print(f'  MAPS:      {maps_time * n_test * 2 / 3600:.1f} saat')

---

## Part 3 — Prompt Engineering

### Part 3.D.1 — Zero-Shot Experiment

In [ ]:
from src.prompts.zero_shot import build_zero_shot_prompt

def run_zero_shot(pairs, direction, model, cfg):
    src_lang, tgt_lang = ('English','Turkish') if direction=='en2tr' else ('Turkish','English')
    src_key = 'en' if direction=='en2tr' else 'tr'
    results = []
    for p in tqdm(pairs, desc=f'Zero-shot {direction}'):
        sys_, usr = build_zero_shot_prompt(p[src_key], src_lang, tgt_lang)
        hyp = model.generate(sys_, usr)
        results.append({'src': p[src_key], 'ref': p['tr' if direction=='en2tr' else 'en'], 'hyp': hyp})
    return results

results_dir = cfg['paths']['results_dir']

zs_en2tr = run_zero_shot(splits['test_en2tr'], 'en2tr', model, cfg)
zs_tr2en = run_zero_shot(splits['test_tr2en'], 'tr2en', model, cfg)

# Checkpoint
for name, data in [('zero_shot_en2tr', zs_en2tr), ('zero_shot_tr2en', zs_tr2en)]:
    with open(f'{results_dir}/{name}.jsonl', 'w', encoding='utf-8') as f:
        for r in data:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

print('Zero-shot tamamlandı.')

### Part 3.D.2 — MAPS Experiment

In [ ]:
from src.prompts.maps_strategy import MAPSPipeline
import dataclasses

maps_pipeline = MAPSPipeline(model, cfg)

def run_maps(pairs, direction, pipeline):
    src_lang, tgt_lang = ('English','Turkish') if direction=='en2tr' else ('Turkish','English')
    src_key = 'en' if direction=='en2tr' else 'tr'
    results = []
    for p in tqdm(pairs, desc=f'MAPS {direction}'):
        r = pipeline.run(p[src_key], src_lang, tgt_lang)
        results.append({
            'src': r.src, 'ref': p['tr' if direction=='en2tr' else 'en'],
            'hyp': r.final_translation,
            'selected_index': r.selected_index,
            'candidates': r.candidates,
            'keywords': r.keywords, 'topic': r.topic,
        })
    return results

maps_en2tr = run_maps(splits['test_en2tr'], 'en2tr', maps_pipeline)
maps_tr2en = run_maps(splits['test_tr2en'], 'tr2en', maps_pipeline)

for name, data in [('maps_en2tr', maps_en2tr), ('maps_tr2en', maps_tr2en)]:
    with open(f'{results_dir}/{name}.jsonl', 'w', encoding='utf-8') as f:
        for r in data:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

print('MAPS tamamlandı.')

---

## Part 4 — RAG

### Part 4.B — RAG Index Oluşturma

In [ ]:
from src.rag.embedder import Embedder
from src.rag.indexer import build_index, save_index, load_index

index_dir = cfg['paths']['index_dir']
embedder = Embedder(cfg['rag']['embedding_model'], cache_dir=cfg['paths']['cache_dir'])

# EN→TR için index: İngilizce cümleler embed edilir
if not Path(f'{index_dir}/en2tr.faiss').exists():
    print('EN→TR index oluşturuluyor...')
    en_texts = [p['en'] for p in splits['train']]
    en_embs = embedder.embed_passages(en_texts, batch_size=cfg['rag']['batch_size_embed'])
    index_en = build_index(en_embs)
    save_index(index_en, splits['train'], index_dir, 'en2tr')
else:
    print('EN→TR index diskten yükleniyor...')
    index_en, _ = load_index(index_dir, 'en2tr')

# TR→EN için index: Türkçe cümleler embed edilir
if not Path(f'{index_dir}/tr2en.faiss').exists():
    print('TR→EN index oluşturuluyor...')
    tr_texts = [p['tr'] for p in splits['train']]
    tr_embs = embedder.embed_passages(tr_texts, batch_size=cfg['rag']['batch_size_embed'])
    index_tr = build_index(tr_embs)
    save_index(index_tr, splits['train'], index_dir, 'tr2en')
else:
    print('TR→EN index diskten yükleniyor...')
    index_tr, _ = load_index(index_dir, 'tr2en')

print('Index hazır.')

### Part 4.B — RAG Experiment

In [ ]:
from src.rag.retriever import Retriever
from src.rag.indexer import load_index
from src.prompts.rag_few_shot import build_rag_prompt

_, train_pairs_en = load_index(index_dir, 'en2tr')
_, train_pairs_tr = load_index(index_dir, 'tr2en')

retriever_en = Retriever(index_en, train_pairs_en, top_k=cfg['rag']['top_k'])
retriever_tr = Retriever(index_tr, train_pairs_tr, top_k=cfg['rag']['top_k'])

def run_rag(pairs, direction, retriever, embedder, model, cfg):
    src_lang, tgt_lang = ('English','Turkish') if direction=='en2tr' else ('Turkish','English')
    src_key = 'en' if direction=='en2tr' else 'tr'
    results = []
    for p in tqdm(pairs, desc=f'RAG {direction}'):
        src = p[src_key]
        q_emb = embedder.embed_queries([src])
        retrieved = retriever.retrieve(q_emb)
        sys_, usr = build_rag_prompt(src, src_lang, tgt_lang, retrieved)
        hyp = model.generate(sys_, usr)
        results.append({
            'src': src,
            'ref': p['tr' if direction=='en2tr' else 'en'],
            'hyp': hyp,
            'retrieved_scores': [r['score'] for r in retrieved],
        })
    return results

rag_en2tr = run_rag(splits['test_en2tr'], 'en2tr', retriever_en, embedder, model, cfg)
rag_tr2en = run_rag(splits['test_tr2en'], 'tr2en', retriever_tr, embedder, model, cfg)

for name, data in [('rag_en2tr', rag_en2tr), ('rag_tr2en', rag_tr2en)]:
    with open(f'{results_dir}/{name}.jsonl', 'w', encoding='utf-8') as f:
        for r in data:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

print('RAG tamamlandı.')

---

## Part 5 — Experimental Comparison

In [ ]:
from src.evaluation.comet_scorer import load_comet_model, score_direction

comet_model = load_comet_model(
    cfg['evaluation']['comet_model'],
    cache_dir=cfg['paths']['cache_dir'],
)

In [ ]:
import pandas as pd

comet_results = {}
batch = cfg['evaluation']['comet_batch_size']

experiments = [
    ('zero_shot', 'en2tr', zs_en2tr,   splits['test_en2tr']),
    ('zero_shot', 'tr2en', zs_tr2en,   splits['test_tr2en']),
    ('maps',      'en2tr', maps_en2tr, splits['test_en2tr']),
    ('maps',      'tr2en', maps_tr2en, splits['test_tr2en']),
    ('rag',       'en2tr', rag_en2tr,  splits['test_en2tr']),
    ('rag',       'tr2en', rag_tr2en,  splits['test_tr2en']),
]

for method, direction, outputs, pairs in experiments:
    hyps = [r['hyp'] for r in outputs]
    result = score_direction(comet_model, pairs, hyps, direction, batch_size=batch)
    key = f'{method}_{direction}'
    comet_results[key] = result['system_score']
    print(f'{key:30s}: COMET = {result["system_score"]:.4f}')

# Sonuçları kaydet
with open(f'{results_dir}/comet_scores.json', 'w') as f:
    json.dump(comet_results, f, indent=2)

# Tablo
rows = []
for direction in ['en2tr', 'tr2en']:
    rows.append({
        'Direction': 'EN→TR' if direction=='en2tr' else 'TR→EN',
        'Zero-Shot': round(comet_results[f'zero_shot_{direction}'], 4),
        'MAPS':      round(comet_results[f'maps_{direction}'], 4),
        'RAG 5-shot':round(comet_results[f'rag_{direction}'], 4),
    })

df = pd.DataFrame(rows)
print('\n', df.to_string(index=False))

In [ ]:
# Örnek çeviri karşılaştırması (rapor için)
idx = 0
p = splits['test_en2tr'][idx]
print(f'Source (EN): {p["en"]}')
print(f'Reference:   {p["tr"]}')
print(f'Zero-shot:   {zs_en2tr[idx]["hyp"]}')
print(f'MAPS:        {maps_en2tr[idx]["hyp"]}')
print(f'RAG 5-shot:  {rag_en2tr[idx]["hyp"]}')